# 实验

### 设置

In [ ]:
# 您可以在代码中直接设置
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

这是我们在整个课程中一直使用的 RAG 应用程序

In [ ]:
import os
import tempfile
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.sitemap import SitemapLoader
from langchain_community.vectorstores import SKLearnVectorStore
from langchain_openai import OpenAIEmbeddings
from langsmith import traceable
from openai import OpenAI
from typing import List
import nest_asyncio

# TODO: 配置这个模型！
MODEL_NAME = "gpt-4o"
MODEL_PROVIDER = "openai"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """您是一个问答任务的助手。
使用以下检索到的上下文片段来回答对话中的最新问题。
如果您不知道答案，请直接说您不知道。
最多使用三句话，保持答案简洁。
"""

openai_client = OpenAI()

def get_vector_db_retriever():
    persist_path = os.path.join(tempfile.gettempdir(), "union.parquet")
    embd = OpenAIEmbeddings()

    # 如果向量存储存在，则加载它
    if os.path.exists(persist_path):
        vectorstore = SKLearnVectorStore(
            embedding=embd,
            persist_path=persist_path,
            serializer="parquet"
        )
        return vectorstore.as_retriever(lambda_mult=0)

    # 否则，索引 LangSmith 文档并创建新的向量存储
    ls_docs_sitemap_loader = SitemapLoader(web_path="https://docs.smith.langchain.com/sitemap.xml", continue_on_failure=True)
    ls_docs = ls_docs_sitemap_loader.load()

    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=500, chunk_overlap=0
    )
    doc_splits = text_splitter.split_documents(ls_docs)

    vectorstore = SKLearnVectorStore.from_documents(
        documents=doc_splits,
        embedding=embd,
        persist_path=persist_path,
        serializer="parquet"
    )
    vectorstore.persist()
    return vectorstore.as_retriever(lambda_mult=0)

nest_asyncio.apply()
retriever = get_vector_db_retriever()

"""
retrieve_documents
- 根据用户的问题从向量存储中返回获取的文档
"""
@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

"""
generate_response
- 在格式化输入后调用 `call_openai` 来生成模型响应
"""
@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"上下文: {formatted_docs} \n\n 问题: {question}"
        }
    ]
    return call_openai(messages)

"""
call_openai
- 从 OpenAI 返回聊天完成输出
"""
@traceable(
    run_type="llm",
    metadata={
        "ls_provider": MODEL_PROVIDER,
        "ls_model_name": MODEL_NAME
    }
)
def call_openai(messages: List[dict]) -> str:
    return openai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )

"""
langsmith_rag
- 调用 `retrieve_documents` 来获取文档
- 调用 `generate_response` 来基于获取的文档生成响应
- 返回模型响应
"""
@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

### 实验

这是一个代码片段，应该与您从启动代码中看到的内容相似！

这里有几个重要组件。

1. 我们已经定义了一个评估器
2. 我们使用目标函数将数据集示例（dict）的形状转换为我们的函数 `langsmith_rag` 接受的输入形状（str）

In [ ]:
from langsmith import evaluate, Client

client = Client()
dataset_name = "RAG应用黄金数据集"

def is_concise_enough(reference_outputs: dict, outputs: dict) -> dict:
    score = len(outputs["output"]) < 1.5 * len(reference_outputs["output"])
    return {"key": "is_concise", "score": int(score)}

def target_function(inputs: dict):
    return langsmith_rag(inputs["question"])

evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="gpt-4o"
)

### 修改您的应用程序

现在，让我们将模型更改为 gpt-35-turbo，看看它的表现如何！

进行此更改，然后运行此代码片段！

In [ ]:
from langsmith import evaluate, Client
from langsmith.schemas import Example, Run

def target_function(inputs: dict):
    return langsmith_rag(inputs["question"])

evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="gpt-3.5-turbo"
)

### 在不同数据片段上运行

##### 数据集版本

您可以通过在 `list_examples` 中使用 `as_of` 参数在 SDK 中对数据集的特定版本执行实验

让我们尝试只在初始数据集上运行。

In [ ]:
evaluate(
    target_function,
    data=client.list_examples(dataset_name=dataset_name, as_of="初始数据集"),   # 我们使用 as_of 来指定版本
    evaluators=[is_concise_enough],
    experiment_prefix="初始数据集版本"
)

##### 数据集分割

您可以在数据集的特定分割上运行实验，让我们尝试在关键示例分割上运行。

In [ ]:
evaluate(
    target_function,
    data=client.list_examples(dataset_name=dataset_name, splits=["关键示例"]),  # 我们传入分割列表
    evaluators=[is_concise_enough],
    experiment_prefix="关键示例分割"
)

##### 特定数据点

您也可以指定要运行实验的单个数据点

In [ ]:
evaluate(
    target_function,
    data=client.list_examples(
        dataset_name=dataset_name, 
        example_ids=[   # 我们传入特定的 example_ids 列表
            # TODO: 您需要粘贴您自己的示例 ID 才能使其工作！
            "",
            ""
        ]
    ),
    evaluators=[is_concise_enough],
    experiment_prefix="两个特定示例 ID"
)

### 其他参数

##### 重复次数

您可以多次运行实验以确保结果一致

In [ ]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="两次重复",
    num_repetitions=2   # 此字段默认为 1
)

##### 并发
您还可以启动并发执行线程以使实验更快完成！

In [ ]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="并发",
    max_concurrency=3,  # 这默认为 None，所以这是一个改进！
)

##### 元数据

您可以（并且应该）向实验添加元数据，以便在 UI 中更容易找到它们

In [ ]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="已添加元数据",
    metadata={  # 我们可以为实验传递自定义元数据，例如模型名称
        "model_name": MODEL_NAME
    }
)